In the previous chapter, we learned that:

In [ ]:
loss.backward()

computes the **gradient for every trainable weight** in the model.

---

# What is a Tensor?

A tensor is the fundamental data structure in PyTorch.

Everything in a neural network is stored as tensors:

- Inputs
- Outputs
- Embeddings
- Weight matrices
- Bias vectors
- Activations
- Gradients
- Loss

A tensor is much more than just a matrix of numbers.

It also stores metadata that allows PyTorch's automatic differentiation (Autograd) system to work.

---

# A Tensor's Internal Structure

Conceptually, a tensor looks like this:

```text
Tensor
├── Values
├── Shape
├── Data Type (dtype)
├── Device (CPU/GPU)
├── requires_grad
├── grad
├── grad_fn
└── Other memory/layout information
```

Let's understand each field.

---

# 1. Values

These are the actual numbers stored inside the tensor.

Example:

In [ ]:
tensor([
    [0.52, -0.18],
    [1.31,  0.74]
])

For a weight matrix, these numbers are the parameters learned during training.

---

# 2. Shape

The tensor stores its dimensions.

Example

In [ ]:
torch.Size([2, 2])

or

In [ ]:
torch.Size([50257, 768])

PyTorch always knows the dimensions of every tensor.

---

# 3. Data Type (dtype)

This specifies what kind of numbers are stored.

Examples

In [ ]:
torch.float32
torch.float64
torch.int64
torch.bool

Neural networks almost always use floating-point tensors.

---

# 4. Device

Every tensor knows where it lives.

Example

In [ ]:
cpu

or

In [ ]:
cuda:0

This allows PyTorch to perform computations either on the CPU or GPU.

---

# 5. requires_grad

This tells Autograd whether this tensor should participate in gradient computation.

Example

In [ ]:
requires_grad = True

Trainable parameters such as weight matrices have

In [ ]:
requires_grad=True

because we want to learn them.

Most intermediate or constant tensors do not require gradients.

---

# 6. grad

This is where PyTorch stores the computed gradients.

Before backpropagation:

```text
Weight Tensor

Values

[
 [0.52, -0.18],
 [1.31,  0.74]
]

Gradient

None
```

After

In [ ]:
loss.backward()

the tensor becomes conceptually

```text
Weight Tensor

Values

[
 [0.52, -0.18],
 [1.31,  0.74]
]

Gradient

[
 [+0.12, -0.08],
 [+0.43, +0.01]
]
```

Notice that the **weight values do not change**.

Only the gradient field gets filled.

---

# Gradient Matrix

Every trainable tensor has a gradient tensor of exactly the same shape.

Example

Weight Matrix

```text
[
 [0.52, -0.18],
 [1.31,  0.74]
]
```

Gradient Matrix

```text
[
 [+0.12, -0.08],
 [+0.43, +0.01]
]
```

Every value has a corresponding gradient.

```text
Weight                  Gradient

0.52        ←→          +0.12

-0.18       ←→          -0.08

1.31        ←→          +0.43

0.74        ←→          +0.01
```

Each gradient answers one question:

> **If this weight changes slightly, how will the loss change?**

Mathematically,

```text
Gradient = ∂Loss / ∂Weight
```

The optimizer later uses these values to update the weights.

---

# What Does a Gradient Mean?

Suppose one weight is

```text
0.82
```

After backpropagation,

```text
Gradient = +0.35
```

This means

> Increasing this weight slightly will increase the loss.

The optimizer therefore decreases this weight.

Another weight may have

```text
Gradient = -0.18
```

This means

> Increasing this weight slightly will decrease the loss.

The optimizer therefore increases this weight.

The gradient itself is **not updated**.

It is simply information used by the optimizer.

---

# 7. grad_fn

Every tensor created through an operation remembers **how it was created**.

Example

In [ ]:
x = torch.tensor([2.0], requires_grad=True)

y = x * 3

Now

In [ ]:
x.grad_fn

returns

```text
None
```

because `x` was created directly.

But

In [ ]:
y.grad_fn

returns something like

```text
<MulBackward0>
```

This tells PyTorch that `y` was produced by a multiplication operation.

Every operation performed during the forward pass creates another node in the computation graph.

---

# The Computation Graph

Suppose we execute

In [ ]:
logits = model(inputs)

loss = criterion(logits, targets)

The graph conceptually looks like

```text
Model Weights
      │
      ▼
Forward Operations
      │
      ▼
Logits
      │
      ▼
Cross Entropy
      │
      ▼
Loss
```

Each tensor remembers the operation that created it.

These connections form the computation graph.

---

# Chain Rule

Backpropagation is based on the **Chain Rule** from calculus.

Instead of computing

```text
∂Loss / ∂Weight
```

directly,

PyTorch computes gradients one operation at a time while moving backward through the computation graph.

Conceptually,

```text
Loss
 ↑
CrossEntropyBackward
 ↑
LinearBackward
 ↑
MatMulBackward
 ↑
EmbeddingBackward
 ↑
Weights
```

Each operation:

1. receives the gradient from the layer above,
2. computes gradients for its own inputs,
3. passes those gradients to the previous operation.

By repeatedly applying the chain rule, PyTorch eventually computes

```text
∂Loss / ∂Weight
```

for every trainable parameter.

---

# Gradient Accumulation

One important property of PyTorch is that gradients **accumulate**.

Suppose after one backward pass

```text
weight.grad

[
 [0.20, 0.15]
]
```

Calling

In [ ]:
loss.backward()

again **without clearing gradients** results in

```text
weight.grad

[
 [0.40, 0.30]
]
```

The new gradients are **added** to the existing ones.

PyTorch does **not** automatically replace old gradients.

This behavior is useful when gradients from multiple mini-batches need to be accumulated before updating the weights.

---

# Clearing Gradients

Before computing gradients for a new training iteration, the previous gradients must be removed.

This is done using

In [ ]:
optimizer.zero_grad()

Conceptually

Before

```text
weight.grad

[
 [0.20, 0.15]
]
```

After

In [ ]:
optimizer.zero_grad()

```text
weight.grad

[
 [0.00, 0.00]
]
```

Now the next call to

In [ ]:
loss.backward()

stores only the gradients for the current forward pass.

A typical training iteration therefore follows this sequence:

In [ ]:
optimizer.zero_grad()

loss = model(...)

loss.backward()

optimizer.step()

---

# Summary

- A tensor stores much more than just numerical values.
- Every trainable tensor can store a gradient tensor in its `.grad` field.
- The gradient tensor always has the same shape as the original tensor.
- Every gradient corresponds one-to-one with a weight.
- Each gradient tells how changing that specific weight would affect the loss.
- Tensors created by operations remember how they were created through `grad_fn`.
- These connections form the computation graph.
- During backpropagation, PyTorch applies the chain rule to compute gradients for every trainable parameter.
- Gradients accumulate by default.
- `optimizer.zero_grad()` clears old gradients before the next backward pass.